# Evaluation
## Ground truth
In agentic RAG, the AI assistant has at its disposal a search tool to help answer user prompts. The system makes an autonomous decision to query an external data source for additional information before prompting the LLM. It is up to the agent to determine how to invoke the search tool, and which arguments to be passed. The quality of this search mechanism has a significant impact on the system's outputs, and must be systematically evaluated.

We want the evaluation system to assess two aspects:
1. the relevance of the information retrieved from the external data source
2. the quality of the output produced by the system based on the search results

There are different approaches for collecting data and performing system evaluation. For one, we can collect logs from user interactions with the agent during dedicated testing sessions. Developers would then review the responses to assess whether the agent and search tool behave as expected. This can be time-consuming and relies heavily on human intervention.

An alternative approach builds a reference dataset, known as _ground truth_, by reverse-engineering the search process. For every document in the external data source, we carefully prompt an LLM to generate a set of possible queries. From this, we obtain a set of question-document pairs (Q<sup>i</sup>, D<sup>i</sup>) that constitutes the input-output mapping we want the agentic system be able to reproduce. Instead of relying on human review and assessment, we trust an LLM (and a carefully crafted prompt) to build a benchmark that the agent and search function are expected to match.

In [1]:
from ingest import load_faq_data
faq_data = load_faq_data()

In [2]:
faq_data[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [3]:
# filter llm-zoomcamp related documents
documents = []

for doc in faq_data:
  if doc["course"] == "llm-zoomcamp":
    documents.append(doc)

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


### Single document

In [5]:
# create a class for enforcing structured model outputs
from pydantic import BaseModel

class Questions(BaseModel):
	questions: list[str]

In [6]:
# create a prompt for generating relevant questions from a document
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
from evaluation_utils import llm_structured

In [ ]:
# use a document as prompt to generate a set of questions
result, usage = llm_structured(
	openai_client,
	data_gen_instructions,
	user_prompt,
	Questions
)

In [11]:
# create a set of question-document pairs
records = []

for q in result.questions:
	records.append({
		"question": q,
		"document": doc["id"]
	})

In [12]:
records

[{'question': 'I just found this course late — can I still join and catch up?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course started, is it still possible to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit the project before the submission window closes to get certified?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course now even though I missed the beginning?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline situation if I want the course certificate after joining late?',
  'document': '74eb249bbf'}]

### All documents

In [13]:
from evaluation_utils import llm_structured_retry

In [14]:
# generates a set of questions from the input document
def generate_ground_truth(doc):
	user_prompt = json.dumps(doc)

	out, usage = llm_structured_retry(
		openai_client,
		data_gen_instructions,
		user_prompt,
		Questions
	)

	results = []

	for q in out.questions:
		results.append({
			"question": q,
			"document": doc["id"]
		})

	return results, usage

In [15]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress
import os
import pandas as pd

In [16]:
# create the full dataset of question-document pairs
if not os.path.exists('./data/ground_truth-new.csv'):
	with ThreadPoolExecutor(max_workers=6) as pool:
		results = map_progress(pool, documents, generate_ground_truth)

	ground_truth = []
	usages = []

	for records, usage in results:
		ground_truth.extend(records)
		usages.append(usage)

	df_ground_truth = pd.DataFrame(ground_truth)
	df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)
else:
	df_ground_truth = pd.read_csv("data/ground_truth-new.csv")


In [17]:
df_ground_truth.head()

,question,document
0,I just found this course — is it too late to j...,74eb249bbf
1,Can I still enroll if the course already started?,74eb249bbf
2,"If I join the course late, can I still get the...",74eb249bbf
3,What’s the deadline for submitting the project...,74eb249bbf
4,Do I need to turn in my project before submiss...,74eb249bbf


## Relevance matrix
At this point, we have compiled a dataset of input-output pairs (Q<sup>i</sup>, D<sup>i</sup>) that we consider to be the _ground truth_, i.e. the target behavior for the agentic system’s search functionality. Every Q<sup>i</sup> is an LLM-generated query related to document D<sup>i</sup> pulled from the external database.

The next step in the evaluation process is to observe the system's current behavior on the same set of queries. For that, we feed every Q<sup>i</sup> as input to the agent, and collect the corresponding output from the system’s search tool. This output A<sup>i</sup> is a set of documents from the external data source that the search engine identified as the most relevant to query Q<sup>i</sup>.

We consider that the search tool was effective if collection A<sup>i</sup> includes document D<sup>i</sup>.

In [ ]:
from ingest import build_index

# fit a text search index
index = build_index(documents)

In [ ]:
# return matching results based on text search
def text_search(query):
	boost_dict = {"question": 3.0, "section": 0.5}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict
	)

### Single document
Our reverse-engineered data generation process has yielded one 'correct' document D<sup>i</sup> for each question Q<sup>i</sup>, i.e. the _ground truth_. Our aim is for the agent system's to include this document as part of its result set when Q<sup>i</sup> is passed as input.

Note that the search engine orders its result by relevance (decreasing). Ideally, we would like the tool to not only include the target document, but to also give it a relatively high relevance score. This would mean that the document is among the top results in the set.

In [ ]:
# isolate a single record from the dataset
ground_truth = df_ground_truth.to_dict(orient="records")
q = ground_truth[0]

In [ ]:
# use the question from the record as input to the search function
doc_id = q["document"]
results = text_search(query=q["question"])

In [26]:
# check that the document from the record is included in the result set
for d in results:
	print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
85384a18e5 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
0fab61eca2 == 74eb249bbf: False
86d99bbf21 == 74eb249bbf: False


For every result set based on query Q<sup>i</sup>, we can generate a vector of binary values to flag whether an element in the set corresponds to the target document D<sup>i</sup>. Each vector will have at most one positive value corresponding to the target document D<sup>i</sup>. As mentioned earlier, the position of the positive value is important in the evaluation.

```
[0, 0, 0, 0, 0] => bad
[0, 0, 1, 0, 0] => good
[1, 0, 0, 0, 0] => better
```

In [ ]:
def compute_relevance(q, search_function):
	# perform search based on the input query
	doc_id = q["document"]
	results = search_function(query=q["question"])

	# compute relevance vector
	relevance = [int(d["id"] == doc_id) for d in results]

	return relevance

By stacking all vectors derived from each query in the dataset, we get a binary _relevance matrix_ where each row _i_ corresponds to the result set returned by the search function for query Q<sup>i</sup>

In [ ]:
from tqdm.auto import tqdm

def compute_relevance_total(ground_truth, search_function):
	relevance_total = []

	for q in tqdm(ground_truth):
		relevance = compute_relevance(q, search_function)
		relevance_total.append(relevance)

	return relevance_total

In [38]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total(ground_truth_sample, text_search)

  0%|          | 0/15 [00:00<?, ?it/s]

In [39]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

## Evaluation metrics
Several metrics can be computed from the derived relevance matrix.

### Hit rate
With __hit rate__ we compute the number of instances where the search tool correctly included document D<sup>i</sup> as part of its result set for query <sup>Q</sup>. This is expressed as a percentage of the total number of points in the dataset.

```
hit rate % = 100 x (# of vectors that include the target document / # of vectors total)
```

In [40]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/665 [00:00<?, ?it/s]

In [48]:
def hit_rate(relevance):
	cnt = 0

	for line in relevance:
		if 1 in line:
				cnt = cnt + 1

	return cnt / len(relevance)

In [49]:
hit_rate(relevance_total)

0.8345864661654135

### Mean reciprocal rank
The drawback of the hit rate metric is that it does not take into account the position of the target document in the result set. For robustness, we prefer a result set that assigns the target document the highest relevance score. We refer to the position of the target document as the vector's _rank_.

__Mean reciprocal rank__ (MRR) is a rank-based evaluation matrix that assigns a score between zero and 1 to every result result set vector in the dataset by computing the reciprocal of each vector's rank. These values are then averaged to obtain a single metric.

In [50]:
def mrr(relevance):
	total_score = 0.0

	for line in relevance:
		for rank in range(len(line)):
			if line[rank] == 1:
				total_score = total_score + 1 / (rank + 1)
				break

	return total_score / len(relevance)

In [51]:
mrr(relevance_total)

0.7161654135338342

### Putting it all together
For a holistic approach, we may want to include several metrics as part of the evaluation.

In [52]:
def evaluate(ground_truth, search_function):
	relevance_total = compute_relevance_total(ground_truth, search_function)

	return {
		"hit_rate": hit_rate(relevance_total),
		"mrr": mrr(relevance_total),
	}

In [53]:
evaluate(
	ground_truth,
	text_search
)

  0%|          | 0/665 [00:00<?, ?it/s]

{'hit_rate': 0.8345864661654135, 'mrr': 0.7161654135338342}

## Performance tuning
### Parameter tuning

In [56]:
def search_boosts(query, question_boost=1.0, answer_boost=1.0, section_boost=1.0):
	boost_dict = {
		"question": question_boost,
		"section": section_boost,
		"answer": answer_boost,
	}

	return index.search(
		query,
		num_results=5,
		boost_dict=boost_dict,
	)

In [58]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
	for answer_boost in [1.0, 2.0, 4.0, 10.0]:
		for section_boost in [0.1, 0.2, 0.5]:
			print(
				f"Evaluating question_boost={question_boost},"
				f" answer_boost={answer_boost},"
				f" section_boost={section_boost}..."
			)

			result = evaluate(
				ground_truth,
				lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
					query,
					question_boost,
					answer_boost,
					section_boost
				)
			)

			results.append({
				"question": question_boost,
				"answer": answer_boost,
				"section": section_boost,
				"hit_rate": result["hit_rate"],
				"mrr": result["mrr"],
			})

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/665 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/665 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/665 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/665 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/665 [00:00<?, ?it/s]

In [59]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.963910,0.862807
8,1.0,4.0,0.5,0.956391,0.860702
6,1.0,4.0,0.1,0.959398,0.860201
23,2.0,10.0,0.5,0.962406,0.857093
35,5.0,10.0,0.5,0.951880,0.856216
19,2.0,4.0,0.2,0.951880,0.856216
3,1.0,2.0,0.1,0.951880,0.856216
4,1.0,2.0,0.2,0.953383,0.856115
20,2.0,4.0,0.5,0.953383,0.855614
18,2.0,4.0,0.1,0.951880,0.854612
